# Step 12 — MCP (Model Context Protocol)

🇬🇧 **English** (this notebook)

MCP is an open standard (introduced by Anthropic in late 2024) for connecting AI agents to external tools and data sources through a common protocol, instead of every framework needing its own bespoke integration for every tool. Where step 11's `SerperDevTool` is a tool built *into* `crewai_tools`, an MCP server is a **separate process** — potentially written in any language, by anyone — that exposes tools over a standard interface. CrewAI connects to it and uses whatever tools it offers.

## Background

> Anthropic (2024). *Introducing the Model Context Protocol*. [modelcontextprotocol.io](https://modelcontextprotocol.io)

Think of MCP like a USB-C port for AI applications: a standardized way to plug an agent into any compliant server exposing tools, data, or workflows — the agent doesn't need custom integration code per tool, and the server doesn't need to know anything about which AI framework is calling it.

## The exercise

This notebook spawns a small, official reference MCP server (`mcp-server-fetch` — fetches and reads web pages) as a local subprocess via `uvx`, and connects the agent to it. No extra installation needed — `uvx` downloads and runs the server on first use (it's part of this project's `uv` toolchain already).

**Before running the cell below**, warm up that download from a terminal (not this notebook) so you can see it happen instead of watching a silent cell:
```bash
uvx mcp-server-fetch --help
```
On a machine with no matching prebuilt wheel for one of its dependencies (e.g. `cryptography`), `uv` falls back to compiling it from source — this took **about 2 minutes** with zero output when we tested it, which looks exactly like a hung cell if you only see it inside Jupyter. Once this command finishes once, the result is cached and every future run (including inside the notebook) starts in a couple of seconds.

Same Researcher role again — but instead of searching broadly like step 5a, it fetches and quotes one specific, canonical source directly: the European Commission's official AI Act page.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent
from crewai.mcp import MCPServerStdio

load_dotenv()

# ── The MCP server — a separate process, connected over stdio ────────────────
fetch_server = MCPServerStdio(command="uvx", args=["mcp-server-fetch"])

# ── Same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent = Agent(
    role=role,
    goal=goal,
    backstory=backstory,
    mcps=[fetch_server],
    verbose=True,
)

# ── Ask it to fetch and use a specific URL ────────────────────────────────────
url = "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai"
question = (
    f"Fetch {url} and list the exact obligations and dates it states for high-risk "
    "AI systems, quoting the page directly rather than paraphrasing from memory."
)

result = await agent.kickoff(question)
print(result.raw)

╭─────────────────────────────────────────────── 🔌 MCP Connection ───────────────────────────────────────────────╮
│                                                                                                                 │
│  MCP Connection Started                                                                                         │
│                                                                                                                 │
│  Transport: stdio                                                                                               │
│  Timeout: 30s                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── ✅ MCP Connected ────────────────────────────────────────────────╮
│                                                                                                                 │
│  MCP Connection Completed                                                                                       │
│                                                                                                                 │
│  Server: uvx mcp-server-fetch                                                                                   │
│  Transport: stdio                                                                                               │
│  Duration: 1747.11ms                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  id:                                                                                                            │
│  7a569808-9249-4a6e-af52-ff78d39980f8                                                                           │
│  role:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  goal:                                                                                                          │
│  Uncover cutting-edge developments in EU AI Act                                                                 │
│  backstory:                                                                                                     │
│  You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act. Known for your  │
│  ability to find the most relevant information and present it in a clear and concise manner.                    │
│  tools:                                                                                                         │
│  [MCPNativeTool(name='uvx_mcp-server-fetch_fetch', description='Tool Name: uvx_mcp_server_fetch_fetch\nTool     │
│  Arguments: {\n  "properties": {\n    "url": {\n      "description": "URL to fetch",\n      "title": "Url",\n   │
│  "type": "string"\n    },\n    "max_length": {\n      "anyOf": [\n        {\n          "type": "integer"\n      │
│  },\n        {\n          "type": "null"\n        }\n      ],\n      "default": null,\n      "description":     │
│  "Maximum number of characters to return.",\n      "title": "Max Length"\n    },\n    "start_index": {\n        │
│  "anyOf": [\n        {\n          "type": "integer"\n        },\n        {\n          "type": "null"\n          │
│  }\n      ],\n      "default": null,\n      "description": "On return output starting at this character index,  │
│  useful if a previous fetch was truncated and more context is required.",\n      "title": "Start Index"\n       │
│  },\n    "raw": {\n      "anyOf": [\n        {\n          "type": "boolean"\n        },\n        {\n            │
│  "type": "null"\n        }\n      ],\n      "default": null,\n      "description": "Get the actual HTML         │
│  content of the requested page, without simplification.",\n      "title": "Raw"\n    }\n  },\n  "required":     │
│  [\n    "url",\n    "max_length",\n    "start_index",\n    "raw"\n  ],\n  "title": "fetchSchema",\n  "type":    │
│  "object",\n  "additionalProperties": false\n}\nTool Description: Fetches a URL from the internet and           │
│  optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and     │
│  were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch     │
│  the most up-to-date information and let the user know that.', env_vars=[], args_schema=<class                  │
│  'crewai.agent.core.fetchSchema'>, description_updated=False, cache_function=<function BaseTool.<lambda> at     │
│  0x10c1ff1a0>, result_as_answer=False, max_usage_count=None, current_usage_count=0)]                            │
│  verbose:                                                                                                       │
│  True                                                  

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: uvx_mcp_server_fetch_fetch                                                                               │
│  Args: {'max_length': 10000, 'raw': False, 'start_index': 0, 'url':                                             │
│  'https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai'}                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Tool uvx_mcp_server_fetch_fetch executed with result: Contents of https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai:
## A Risk-based Approach

The AI Act defines 4 levels of risk for AI systems:

![pyramid showing the four levels ...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: uvx_mcp_server_fetch_fetch                                                                               │
│  Output: Contents of https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai:                 │
│  ## A Risk-based Approach                                                                                       │
│                                                                                                                 │
│  The AI Act defines 4 levels of risk for AI systems:                                                            │
│                                                                                                                 │
│  ![pyramid showing the four levels of risk: Unacceptable risk; High-risk; limited risk, minimal or no           │
│  risk](https://ec.europa.eu/information_society/newsroom/image/document/2021-17/pyramid_7F5843E5-9386-8052-931  │
│  F5C4E98C6E5F2_75757.jpg)                                                                                       │
│                                                                                                                 │
│  ### Unacceptable risk                                                                                          │
│                                                                                                                 │
│  All AI systems considered a clear threat to the safety, livelihoods and rights of people are banned. The **AI  │
│  Act prohibits eight practices**, namely:                                                                       │
│                                                                                                                 │
│  1. harmful AI-based manipulation and deception                                                                 │
│  2. harmful AI-based exploitation of vulnerabilities                                                            │
│  3. social scoring                                                                                              │
│  4. Individual criminal offence risk assessment or prediction                                                   │
│  5. untargeted scraping of the internet or CCTV material to create or expand facial recognition databases       │
│  6. emotion recognition in workplaces and education institutions                                                │
│  7. biometric categorisation to deduce certain protected characteristics                                        │
│  8. real-time remote biometric identification for law enforcement purposes in publicly accessible spaces        │
│                                                                                                                 │
│  The prohibitions became effective in February 2025. The Commission published 2 key documents to support the    │
│  practical application of the prohibited practices:                                                             │
│                                                                                                                 │
│  * The [guidelines on prohibited AI practices under the AI                                                      │
│  Act](https://digital-strategy.ec.europa.eu/en/library/commission-publishes-guidelines-prohibited-artificial-i  │
│  ntelligence-ai-practices-defined-ai-act), which offer legal explanations and practical examples to help        │
│  stakeholders understand and comply with the prohibitions.                                                      │
│  * The [guidelines on the AI system definition of the A

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the official European Commission page provided, here are the details regarding high-risk AI systems:  │
│                                                                                                                 │
│  ### Obligations for High-Risk AI Systems                                                                       │
│  The page states that **"High-risk AI systems are subject to strict obligations before they can be put on the   │
│  market:"**                                                                                                     │
│                                                                                                                 │
│  *   "adequate risk assessment and mitigation systems"                                                          │
│  *   "high-quality of the datasets feeding the system to minimise risks of discriminatory outcomes"             │
│  *   "logging of activity to ensure traceability of results"                                                    │
│  *   "detailed documentation providing all information necessary on the system and its purpose for authorities  │
│  to assess its compliance"                                                                                      │
│  *   "clear and adequate information to the deployer"                                                           │
│  *   "appropriate human oversight measures"                                                                     │
│  *   "high level of robustness, cybersecurity and accuracy"                                                     │
│                                                                                                                 │
│  ### Dates                                                                                                      │
│  The provided text **does not specify an exact date** for when the obligations for high-risk AI systems become  │
│  effective.                                                                                                     │
│                                                                                                                 │
│  It explicitly mentions dates for other categories, but not for the general high-risk obligations:              │
│  *   **Prohibited practices:** "The prohibitions became effective in February 2025."                            │
│  *   **Transparency rules:** "The transparency rules of the AI Act will come into effect in August 2026."       │
│  *   **General-Purpose AI (GPAI) rules:** "The AI Act rules on GPAI became effective in August 2025."           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on the official European Commission page provided, here are the details regarding high-risk AI systems:

### Obligations for High-Risk AI Systems
The page states that **"High-risk AI systems are subject to strict obligations before they can be put on the market:"**

*   "adequate risk assessment and mitigation systems"
*   "high-quality of the datasets feeding the system to minimise risks of discriminatory outcomes"
*   "logging of activity to ensure traceability of results"
*   "detailed documentation providing all information necessary on the system and its purpose for authorities to assess its compliance"
*   "clear and adequate information to the deployer"
*   "appropriate human oversight measures"
*   "high level of robustness, cybersecurity and accuracy"

### Dates
The provided text **does not specify an exact date** for when the obligations for high-risk AI systems become effective. 

It explicitly mentions dates for other categories, but not for the general high-risk obli

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  id: 7a569808-9249-4a6e-af52-ff78d39980f8                                                                       │
│  role: EU AI Act Senior Data Researcher                                                                         │
│  goal: Uncover cutting-edge developments in EU AI Act                                                           │
│  backstory: You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act.      │
│  Known for your ability to find the most relevant information and present it in a clear and concise manner.     │
│  tools: [MCPNativeTool(name='uvx_mcp-server-fetch_fetch', description='Tool Name:                               │
│  uvx_mcp_server_fetch_fetch\nTool Arguments: {\n  "properties": {\n    "url": {\n      "description": "URL to   │
│  fetch",\n      "title": "Url",\n      "type": "string"\n    },\n    "max_length": {\n      "anyOf": [\n        │
│  {\n          "type": "integer"\n        },\n        {\n          "type": "null"\n        }\n      ],\n         │
│  "default": null,\n      "description": "Maximum number of characters to return.",\n      "title": "Max         │
│  Length"\n    },\n    "start_index": {\n      "anyOf": [\n        {\n          "type": "integer"\n        },\n  │
│  {\n          "type": "null"\n        }\n      ],\n      "default": null,\n      "description": "On return      │
│  output starting at this character index, useful if a previous fetch was truncated and more context is          │
│  required.",\n      "title": "Start Index"\n    },\n    "raw": {\n      "anyOf": [\n        {\n                 │
│  "type": "boolean"\n        },\n        {\n          "type": "null"\n        }\n      ],\n      "default":      │
│  null,\n      "description": "Get the actual HTML content of the requested page, without simplification.",\n    │
│  "title": "Raw"\n    }\n  },\n  "required": [\n    "url",\n    "max_length",\n    "start_index",\n    "raw"\n   │
│  ],\n  "title": "fetchSchema",\n  "type": "object",\n  "additionalProperties": false\n}\nTool Description:      │
│  Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you   │
│  did not have internet access, and were advised to refuse and tell the user this, this tool now grants you      │
│  internet access. Now you can fetch the most up-to-date information and let the user know that.', env_vars=[],  │
│  args_schema=<class 'crewai.agent.core.fetchSchema'>, description_updated=False, cache_function=<function       │
│  BaseTool.<lambda> at 0x10c1ff1a0>, result_as_answer=False, max_usage_count=None, current_usage_count=1)]       │
│  verbose: True                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Your task

1. Run the cell. If you skipped the terminal warm-up above and this is genuinely the first time `mcp-server-fetch` runs on your machine, expect it to sit with **no output at all for a minute or two** while `uv` downloads (and possibly compiles) it in the background — that's normal, not a hang. Check the verbose log: can you see the MCP tool being called, and with what arguments?

2. Compare to step 11: both let the agent reach outside its training data, but 5a searches broadly and summarizes across sources, while this one fetches and quotes exactly one page. Which answer do you trust more, and why?

3. Swap in your own team's topic — including a real URL relevant to it (e.g. the official regulation text, a Wikipedia page, a company's about page).

4. Fill in the **Step 12** section of `EVALUATION.md`.